# SimpleTMG - smartbuilding



In [1]:
!pip install PyWavelets gdown --quiet
import os
import json
import shutil
import subprocess
import pandas as pd
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
REPO_URL = "https://github.com/agamswami/simpleTM-show"
REPO_DIR = "/kaggle/working/SimpleTM"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!sed -i 's/np\.Inf/np.inf/g' utils/tools.py


Cloning into '/kaggle/working/SimpleTM'...
remote: Enumerating objects: 245, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 245 (delta 115), reused 220 (delta 90), pack-reused 0 (from 0)
Receiving objects: 100% (245/245), 25.93 MiB | 34.67 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Working directory: /kaggle/working/SimpleTM


In [3]:
FILE_ID = "1JOgmL2ntyxKg6eB132ytCbVZumiHnDLR"
OUTPUT_PATH = "/kaggle/working/dataset.zip"
DATASET_DIR = "./dataset"

if not os.path.exists(DATASET_DIR):
    !gdown https://drive.google.com/uc?id={FILE_ID} -O {OUTPUT_PATH}
    import zipfile
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(OUTPUT_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    print("Dataset extracted successfully.")
else:
    print("Dataset already exists.")

for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:
        print(f'{subindent}{file}')


Downloading...
From (original): https://drive.google.com/uc?id=1JOgmL2ntyxKg6eB132ytCbVZumiHnDLR
From (redirected): https://drive.google.com/uc?id=1JOgmL2ntyxKg6eB132ytCbVZumiHnDLR&confirm=t&uuid=9f8978c9-15b0-4cc1-b8cd-cec9a2091904
To: /kaggle/working/dataset.zip
100%|████████████████████████████████████████| 172M/172M [00:02<00:00, 77.8MB/s]
Dataset extracted successfully.
dataset/
  SimpleTM_datasets/
    .DS_Store
    smartbuilding/
      smart.csv
    Solar/
      solar_AL.txt
    PEMS/
      PEMS07.npz
      PEMS04.npz
      PEMS08.npz
      PEMS03.npz
    electricity/
      .DS_Store
      electricity.csv
    traffic/
      .DS_Store
      traffic.csv
    weather/
      weather.csv
    ETT-small/
      ETTh2.csv
      ETTm1.csv
      ETTm2.csv
      ETTh1.csv
  __MACOSX/
    ._SimpleTM_datasets
    SimpleTM_datasets/
      ._PEMS
      ._traffic
      ._weather
      ._ETT-small
      ._.DS_Store
      ._electricity
      ._Solar
      Solar/
        ._solar_AL.txt
      PEMS/
 

In [ ]:
SELECTED_DATASETS = None
# Example: SELECTED_DATASETS = ["ETTh1", "ETTm1", "smartbuilding"]

all_datasets = [
    # Official SimpleTM paper script settings for pred_len=96
    # {"name": "ETTh1", "data": "ETTh1", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTh1.csv", "enc_in": 7, "d_model": 32, "d_ff": 32, "e_layers": 1, "wv": "db1", "m": 3, "alpha": 0.3, "learning_rate": 0.02, "batch_size": 256, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "TST", "freq": "h", "itr": 3, "l1_weight": 0.0005},
    # {"name": "ETTh2", "data": "ETTh2", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTh2.csv", "enc_in": 7, "d_model": 32, "d_ff": 32, "e_layers": 1, "wv": "bior3.1", "m": 1, "alpha": 0.1, "learning_rate": 0.006, "batch_size": 256, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "TST", "freq": "h", "itr": 3, "l1_weight": 0.0005},
    # {"name": "ETTm1", "data": "ETTm1", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTm1.csv", "enc_in": 7, "d_model": 32, "d_ff": 32, "e_layers": 1, "wv": "db1", "m": 3, "alpha": 0.1, "learning_rate": 0.02, "batch_size": 256, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "TST", "freq": "t", "itr": 3, "l1_weight": 0.005},
    # {"name": "ETTm2", "data": "ETTm2", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTm2.csv", "enc_in": 7, "d_model": 32, "d_ff": 32, "e_layers": 1, "wv": "bior3.1", "m": 3, "alpha": 0.3, "learning_rate": 0.006, "batch_size": 256, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "TST", "freq": "t", "itr": 3, "l1_weight": 0.0005},
    # {"name": "weather", "data": "custom", "root": "./dataset/SimpleTM_datasets/weather", "path": "weather.csv", "enc_in": 21, "d_model": 32, "d_ff": 32, "e_layers": 4, "wv": "db4", "m": 3, "alpha": 0.3, "learning_rate": 0.01, "batch_size": 256, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "TST", "freq": "h", "itr": 3, "l1_weight": 5e-05},
    # {"name": "electricity", "data": "custom", "root": "./dataset/SimpleTM_datasets/electricity", "path": "electricity.csv", "enc_in": 321, "d_model": 256, "d_ff": 1024, "e_layers": 1, "wv": "db1", "m": 3, "alpha": 0.0, "learning_rate": 0.01, "batch_size": 256, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "TST", "freq": "h", "itr": 1, "l1_weight": 0.0},
    # {"name": "traffic", "data": "custom", "root": "./dataset/SimpleTM_datasets/traffic", "path": "traffic.csv", "enc_in": 862, "d_model": 512, "d_ff": 1024, "e_layers": 2, "wv": "db1", "m": 3, "alpha": 0.1, "learning_rate": 0.003, "batch_size": 24, "train_epochs": 10, "patience": 3, "use_norm": 1, "lradj": "TST", "freq": "h", "itr": 1, "l1_weight": 0.0},
    # {"name": "Solar", "data": "Solar", "root": "./dataset/SimpleTM_datasets/Solar", "path": "solar_AL.txt", "enc_in": 137, "d_model": 128, "d_ff": 256, "e_layers": 1, "wv": "db8", "m": 3, "alpha": 0.0, "learning_rate": 0.01, "batch_size": 256, "train_epochs": 10, "patience": 3, "use_norm": 0, "lradj": "TST", "freq": "h", "itr": 3, "l1_weight": 0.005},
    # {"name": "PEMS03", "data": "PEMS", "root": "./dataset/SimpleTM_datasets/PEMS", "path": "PEMS03.npz", "enc_in": 358, "d_model": 256, "d_ff": 1024, "e_layers": 1, "wv": "bior3.1", "m": 3, "alpha": 0.1, "learning_rate": 0.002, "batch_size": 16, "train_epochs": 20, "patience": 10, "use_norm": 0, "lradj": "TST", "freq": "h", "itr": 3, "l1_weight": 0.005},
    # {"name": "PEMS04", "data": "PEMS", "root": "./dataset/SimpleTM_datasets/PEMS", "path": "PEMS04.npz", "enc_in": 307, "d_model": 256, "d_ff": 1024, "e_layers": 1, "wv": "bior3.1", "m": 3, "alpha": 0.1, "learning_rate": 0.002, "batch_size": 16, "train_epochs": 20, "patience": 10, "use_norm": 0, "lradj": "TST", "freq": "h", "itr": 3, "l1_weight": 5e-05},
    # {"name": "PEMS07", "data": "PEMS", "root": "./dataset/SimpleTM_datasets/PEMS", "path": "PEMS07.npz", "enc_in": 883, "d_model": 256, "d_ff": 512, "e_layers": 1, "wv": "db1", "m": 3, "alpha": 0.1, "learning_rate": 0.002, "batch_size": 16, "train_epochs": 20, "patience": 10, "use_norm": 0, "lradj": "TST", "freq": "h", "itr": 3, "l1_weight": 5e-05},
    # {"name": "PEMS08", "data": "PEMS", "root": "./dataset/SimpleTM_datasets/PEMS", "path": "PEMS08.npz", "enc_in": 170, "d_model": 256, "d_ff": 1024, "e_layers": 1, "wv": "db12", "m": 3, "alpha": 0.0, "learning_rate": 0.001, "batch_size": 16, "train_epochs": 20, "patience": 10, "use_norm": 0, "lradj": "TST", "freq": "h", "itr": 3, "l1_weight": 0.0},

    # No official script exists for smartbuilding, so use one fixed recipe for every variant on this dataset.
    {"name": "smartbuilding", "data": "custom", "root": "./dataset/SimpleTM_datasets/smartbuilding", "path": "smart.csv", "enc_in": 37, "d_model": 64, "d_ff": 128, "e_layers": 1, "wv": "db4", "m": 3, "alpha": 0.3, "learning_rate": 0.003, "batch_size": 64, "train_epochs": 20, "patience": 5, "use_norm": 1, "lradj": "TST", "freq": "h", "itr": 3, "l1_weight": 5e-05, "target": "Floor_Total(kW)"},
]

experiments = [
    {"tag": "SWT_original", "model": "SimpleTM_SWT", "attention_mode": "original"},
    {"tag": "SWT_dual", "model": "SimpleTM_SWT", "attention_mode": "dual"},
    {"tag": "FFT_original", "model": "SimpleTM_FFT", "attention_mode": "original"},
    {"tag": "FFT_dual", "model": "SimpleTM_FFT", "attention_mode": "dual"},
    {"tag": "Conv_original", "model": "SimpleTM_Conv", "attention_mode": "original"},
    {"tag": "Conv_dual", "model": "SimpleTM_Conv", "attention_mode": "dual"},
    {"tag": "Hybrid_original", "model": "SimpleTM_Hybrid", "attention_mode": "original"},
    {"tag": "Hybrid_dual", "model": "SimpleTM_Hybrid", "attention_mode": "dual"},
]

if SELECTED_DATASETS is not None:
    datasets = [ds for ds in all_datasets if ds["name"] in SELECTED_DATASETS]
else:
    datasets = all_datasets

print("Datasets selected:")
for ds in datasets:
    print(f"- {ds['name']}: e_layers={ds['e_layers']}, d_model={ds['d_model']}, d_ff={ds['d_ff']}, lr={ds['learning_rate']}, batch={ds['batch_size']}, itr={ds['itr']}, wv={ds['wv']}, m={ds['m']}, alpha={ds['alpha']}, l1={ds['l1_weight']}")

print("\nExperiments selected:")
for exp in experiments:
    print(f"- {exp['tag']}")

OUTPUT_DIR = "/kaggle/working/SimpleTM_Paper96_AllVariants_Results"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists('./checkpoints'):
    shutil.rmtree('./checkpoints')
if os.path.exists('result_long_term_forecast.txt'):
    os.remove('result_long_term_forecast.txt')


Datasets selected:
- smartbuilding: e_layers=1, d_model=64, d_ff=128, lr=0.003, batch=64, itr=3, wv=db4, m=3, alpha=0.3, l1=5e-05

Experiments selected:
- SWT_original
- SWT_dual
- FFT_original
- FFT_dual
- Conv_original
- Conv_dual
- Hybrid_original
- Hybrid_dual


In [6]:
for ds in datasets:
    print("\n" + "=" * 100)
    print(f"DATASET: {ds['name']} | variates={ds['enc_in']} | official_pred96_recipe")
    print("=" * 100 + "\n")

    for exp in experiments:
        print(f">>> Training {exp['tag']} on {ds['name']} <<<\n")
        unique_model_id = f"{ds['name']}_{exp['model']}_{exp['attention_mode']}"

        cmd = [
            "python", "-u", "run.py",
            "--is_training", "1",
            "--model", exp["model"],
            "--attention_mode", exp["attention_mode"],
            "--model_id", unique_model_id,
            "--data", ds["data"],
            "--root_path", ds["root"],
            "--data_path", ds["path"],
            "--features", "M",
            "--freq", ds["freq"],
            "--seq_len", "96",
            "--pred_len", "96",
            "--e_layers", str(ds["e_layers"]),
            "--d_model", str(ds["d_model"]),
            "--d_ff", str(ds["d_ff"]),
            "--enc_in", str(ds["enc_in"]),
            "--dec_in", str(ds["enc_in"]),
            "--c_out", str(ds["enc_in"]),
            "--wv", ds["wv"],
            "--m", str(ds["m"]),
            "--alpha", str(ds["alpha"]),
            "--l1_weight", str(ds["l1_weight"]),
            "--learning_rate", str(ds["learning_rate"]),
            "--batch_size", str(ds["batch_size"]),
            "--train_epochs", str(ds["train_epochs"]),
            "--patience", str(ds["patience"]),
            "--itr", str(ds["itr"]),
            "--num_workers", "2",
            "--lradj", ds["lradj"],
            "--use_norm", str(ds["use_norm"]),
            "--checkpoints", f"{OUTPUT_DIR}/checkpoints/",
            "--fix_seed", "2025",
            "--des", "Paper96",
        ]

        if "conv_kernel_sizes" in ds:
            cmd.extend(["--conv_kernel_sizes", ds["conv_kernel_sizes"]])

        if "target" in ds:
            cmd.extend(["--target", ds["target"]])

        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        for line in process.stdout:
            print(line, end="")
        process.wait()

        if process.returncode == 0:
            print(f"\nSUCCESS: Finished {exp['tag']} on {ds['name']}\n")
        else:
            raise RuntimeError(f"Run failed for {ds['name']} | {exp['tag']} with exit code {process.returncode}")

print("\nGathering generated PDF plots...")
if os.path.exists('./checkpoints'):
    for root_dir, dirs, files in os.walk('./checkpoints'):
        for file in files:
            if file.endswith('.pdf') or file.endswith('.png'):
                folder_name = os.path.basename(root_dir)
                source_path = os.path.join(root_dir, file)
                target_plot_dir = os.path.join(OUTPUT_DIR, 'plots', folder_name)
                os.makedirs(target_plot_dir, exist_ok=True)
                shutil.copy(source_path, os.path.join(target_plot_dir, file))
    print(f"Copied PDF plots to {OUTPUT_DIR}/plots/")

if os.path.exists('result_long_term_forecast.txt'):
    shutil.copy('result_long_term_forecast.txt', os.path.join(OUTPUT_DIR, 'result_long_term_forecast.txt'))
    print("Copied metric text results to output directory.")



DATASET: smartbuilding | variates=37 | official_pred96_recipe

>>> Training SWT_original on smartbuilding <<<

Args in experiment:
Namespace(is_training=1, model_id='smartbuilding_SimpleTM_SWT_original', model='SimpleTM_SWT', data='custom', root_path='./dataset/SimpleTM_datasets/smartbuilding', data_path='smart.csv', features='M', target='Floor_Total(kW)', freq='h', checkpoints='/kaggle/working/SimpleTM_Paper96_AllVariants_Results/checkpoints/', seq_len=96, label_len=0, pred_len=96, enc_in=37, dec_in=37, c_out=37, n_heads=8, d_layers=1, moving_avg=25, factor=1, distil=True, dropout=0.1, geomattn_dropout=0.5, embed='timeF', activation='gelu', do_predict=False, num_workers=2, itr=3, train_epochs=20, batch_size=64, patience=5, learning_rate=0.003, des='Paper96', loss='MSE', lradj='TST', pct_start=0.2, use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', exp_name='MTSF', channel_independence=False, inverse=False, class_strategy='projection', target_root_path='./data

In [7]:
import re

results_file = os.path.join(OUTPUT_DIR, 'result_long_term_forecast.txt')
metric_order = ['MSE', 'MAE', 'RMSE', 'MAPE', 'WAPE', 'R2']
pattern = re.compile(r'^(?P<dataset>[^_]+)_(?P<model>SimpleTM_(?:SWT|FFT|Conv|Hybrid))_(?P<attention>original|dual)_')
metric_pattern = re.compile(r'([a-zA-Z0-9_]+):([^,]+)')

if os.path.exists(results_file):
    with open(results_file, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    entries = content.split('\n\n') if content else []
    rows = []
    for entry in entries:
        lines = [line.strip() for line in entry.splitlines() if line.strip()]
        if len(lines) < 2:
            continue
        setting = lines[0]
        metrics_line = lines[-1]
        match = pattern.search(setting)
        if not match:
            continue

        model_type = match.group('model')
        attention_mode = match.group('attention')
        tokenization = model_type.replace('SimpleTM_', '')
        row = {
            'Dataset': match.group('dataset'),
            'Model': model_type,
            'AttentionMode': attention_mode,
            'Variant': f'{tokenization}_{attention_mode}',
            'Setting': setting,
        }
        for metric_name, value in metric_pattern.findall(metrics_line):
            try:
                row[metric_name.upper()] = float(value.strip())
            except ValueError:
                pass
        rows.append(row)

    if rows:
        raw_df = pd.DataFrame(rows).sort_values(['Dataset', 'Model', 'AttentionMode', 'Setting']).reset_index(drop=True)
        value_columns = [col for col in metric_order if col in raw_df.columns]
        raw_columns = ['Dataset', 'Model', 'AttentionMode', 'Variant', 'Setting'] + value_columns
        raw_df = raw_df[raw_columns]
        print('Raw runs:')
        print(raw_df.to_string(index=False))

        raw_path = os.path.join(OUTPUT_DIR, 'raw_metrics.csv')
        raw_df.to_csv(raw_path, index=False)

        agg_df = raw_df.groupby(['Dataset', 'Variant'], as_index=False)[value_columns].mean()
        counts_df = raw_df.groupby(['Dataset', 'Variant'], as_index=False).size().rename(columns={'size': 'NumRuns'})
        agg_df = agg_df.merge(counts_df, on=['Dataset', 'Variant'], how='left')
        ordered_columns = ['Dataset', 'Variant'] + value_columns + ['NumRuns']
        agg_df = agg_df[ordered_columns]
        print('\nAveraged results across repeated itr runs:')
        print(agg_df.sort_values(['Dataset', 'Variant']).to_string(index=False))

        averaged_path = os.path.join(OUTPUT_DIR, 'averaged_metrics.csv')
        agg_df.to_csv(averaged_path, index=False)

        pivot_df = agg_df.pivot(index='Dataset', columns='Variant', values=value_columns)
        variant_order = ['SWT_original', 'SWT_dual', 'FFT_original', 'FFT_dual', 'Conv_original', 'Conv_dual', 'Hybrid_original', 'Hybrid_dual']
        pivot_df = pivot_df.reindex(columns=variant_order, level=1)
        final_path = os.path.join(OUTPUT_DIR, 'final_metrics.csv')
        pivot_df.to_csv(final_path)

        print(f'\nSaved raw results to: {raw_path}')
        print(f'Saved averaged results to: {averaged_path}')
        print(f'Saved final table to: {final_path}')
    else:
        print('No valid metrics found in result file.')
else:
    print('Result file not found.')


Raw runs:
      Dataset           Model AttentionMode         Variant                                                                                                                               Setting      MSE      MAE     RMSE     MAPE     WAPE       R2
smartbuilding   SimpleTM_Conv          dual       Conv_dual             smartbuilding_SimpleTM_Conv_dual_SimpleTM_Conv_dual_custom_96_96_64_128_1_db4_None_None_3_0.3_5e-05_0.003_TST_64_2025_1_0 0.289310 0.299444 0.537876 0.463356 0.376986 0.775538
smartbuilding   SimpleTM_Conv          dual       Conv_dual             smartbuilding_SimpleTM_Conv_dual_SimpleTM_Conv_dual_custom_96_96_64_128_1_db4_None_None_3_0.3_5e-05_0.003_TST_64_2025_1_1 0.298590 0.307154 0.546434 0.477419 0.386692 0.768339
smartbuilding   SimpleTM_Conv          dual       Conv_dual             smartbuilding_SimpleTM_Conv_dual_SimpleTM_Conv_dual_custom_96_96_64_128_1_db4_None_None_3_0.3_5e-05_0.003_TST_64_2025_1_2 0.298376 0.306608 0.546238 0.475129 0.386005 0.76850

In [8]:
print("Packing logs, weights, plots, and results...")
archive_path = shutil.make_archive('/kaggle/working/SimpleTM_Paper96_AllVariants_Results', 'zip', OUTPUT_DIR)
print(f"\nDONE! Download {archive_path} from the Kaggle Output pane.")


Packing logs, weights, plots, and results...

DONE! Download /kaggle/working/SimpleTM_Paper96_AllVariants_Results.zip from the Kaggle Output pane.
